In [ ]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

# Local imports
from src.config import config
from src.utils import load_panel_data
from src.data_builder import DataBuilder

# Set matplotlib style
plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'figure.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print("Imports successful!")

## 1. Load and Validate Data

In [ ]:
# Load panel data
data_path = config.DATA_DIR / "df_panel_final.csv"

if not data_path.exists():
    print("Building dataset...")
    builder = DataBuilder()
    df_panel = builder.run()
else:
    df_panel = pd.read_csv(data_path, parse_dates=["date"])

# Country list matching legacy
COUNTRIES = {
    "Australia": "AUS",
    "Brazil": "BRA",
    "Canada": "CAN",
    "Euro Area": "EA",
    "Indonesia": "IND",
    "Japan": "JPN",
    "Korea": "KOR",
    "Mexico": "MEX",
    "New Zealand": "NZL",
    "Philippines": "PHL",
    "Switzerland": "CHE",
    "Thailand": "THA",
    "Türkiye": "TUR",
    "United Kingdom": "GBR"
}

print(f"Loaded data with {len(df_panel)} observations")
print(f"\nCountries in dataset:")
print(df_panel['country'].value_counts().sort_index())
print(f"\nDate range: {df_panel['date'].min()} to {df_panel['date'].max()}")

## 2. Descriptive Statistics Table

In [ ]:
# Compute descriptive statistics for each country
from scipy import stats as sp_stats

desc_stats = []

for country_name in COUNTRIES.keys():
    country_data = df_panel[df_panel['country'] == country_name]
    
    if len(country_data) > 0:
        r_s = country_data['r_s'].dropna()
        
        desc_stats.append({
            'Country': country_name,
            'Mean': r_s.mean(),
            'Variance': r_s.var(),
            'Skewness': sp_stats.skew(r_s),
            'Kurtosis': sp_stats.kurtosis(r_s)
        })
    else:
        desc_stats.append({
            'Country': country_name,
            'Mean': np.nan,
            'Variance': np.nan,
            'Skewness': np.nan,
            'Kurtosis': np.nan
        })

df_desc = pd.DataFrame(desc_stats).set_index('Country')
print("\n=== DESCRIPTIVE STATISTICS (Exchange Rate Returns) ===")
print(df_desc.round(4))

# Save to CSV for LaTeX import
output_dir = config.RESULTS_DIR
output_dir.mkdir(parents=True, exist_ok=True)
df_desc.to_csv(output_dir / "descriptive_statistics.csv")
print(f"\nSaved to: {output_dir / 'descriptive_statistics.csv'}")

## 3. STR Estimation Pipeline

### 3.1 Helper Functions for Z-Candidate Selection

In [ ]:
# Z-candidate generation matching legacy
def build_z_candidates(data: pd.DataFrame) -> Dict[str, pd.Series]:
    """
    Generates 35 transition variable candidates:
    7 base variables × 5 lags each
    """
    z_base = {
        "eta": data["q"],
        "eta_abs": np.abs(data["q"]),
        "drs_abs": np.abs(data["r_s"]),
        "ID": (data["i_for"] - data["i_dom"]),
        "ppp_abs": np.abs(data["f_ppp"]),
        "drf_abs": np.abs(data["f_ppp_rel"]),
        "rel_misalignment_abs": np.abs(data["f_ppp_rel"] - data["r_s"]),
    }
    
    candidates = {}
    for name, series in z_base.items():
        for lag in range(1, 6):  # lags 1-5
            candidates[f"{name}_lag{lag}"] = series.shift(lag)
    
    return candidates

print("Z-candidate helper function defined")

### 3.2 LM Linearity Test Implementation

In [ ]:
import statsmodels.api as sm
from scipy import stats

def lm_linearity_test_for_z(data: pd.DataFrame, z: pd.Series, maxlags_hac: int = 4) -> dict:
    """
    LM linearity test matching legacy implementation.
    Tests H0: Linear vs H1: STR for a specific z variable.
    """
    # Build base linear model
    df = pd.DataFrame({
        "y": data["r_s"] - (data["i_for"] - data["i_dom"]),
        "rs_lag1": data["r_s"].shift(1),
        "eta_lag1": data["q"].shift(1),
    }).dropna()
    
    # Join with z
    df = df.join(z.rename("z"), how="inner").dropna()
    
    if len(df) < 20:
        return None
    
    T = len(df)
    y = df["y"].to_numpy()
    X = df[["rs_lag1", "eta_lag1"]].to_numpy()
    
    # Linear model residuals
    lin_res = sm.OLS(y, X).fit()
    u = lin_res.resid
    
    # Taylor expansion terms
    zc = df["z"].to_numpy()
    z1, z2, z3 = zc, zc**2, zc**3
    
    W_rs = np.column_stack([
        df["rs_lag1"].to_numpy() * z1,
        df["rs_lag1"].to_numpy() * z2,
        df["rs_lag1"].to_numpy() * z3
    ])
    W_eta = np.column_stack([
        df["eta_lag1"].to_numpy() * z1,
        df["eta_lag1"].to_numpy() * z2,
        df["eta_lag1"].to_numpy() * z3
    ])
    W = np.column_stack([W_rs, W_eta])
    k = W.shape[1]
    
    # Project out X
    XTX_inv = np.linalg.pinv(X.T @ X)
    P_X = X @ XTX_inv @ X.T
    M_X = np.eye(T) - P_X
    W_star = M_X @ W
    
    # Auxiliary regression
    aux = sm.OLS(u, W_star).fit()
    R2 = aux.rsquared
    LM = T * R2
    pval = 1.0 - stats.chi2.cdf(LM, df=k)
    
    # HAC version
    aux_hac = sm.OLS(u, W_star).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags_hac})
    beta = np.asarray(aux_hac.params)
    cov = np.asarray(aux_hac.cov_params())
    cov_inv = np.linalg.pinv(cov)
    LM_HAC = float(beta.T @ cov_inv @ beta)
    pval_HAC = 1.0 - stats.chi2.cdf(LM_HAC, df=k)
    
    return {
        "LM": LM,
        "p_value": pval,
        "LM_HAC": LM_HAC,
        "p_value_HAC": pval_HAC,
        "df": k,
        "nobs": T
    }

print("LM linearity test function defined")

### 3.3 STR Estimation for All Countries

In [ ]:
from src.econometrics import STRModel
from scipy.special import expit

# Storage for results
str_results = {}
str_linearity_tests = {}

for country_name, code in COUNTRIES.items():
    print(f"\n{'='*60}")
    print(f"Processing {country_name} ({code})")
    print(f"{'='*60}")
    
    try:
        # Get country data
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        
        if len(country_df) < 50:
            print(f"Insufficient data for {country_name}")
            continue
        
        # Generate z-candidates
        z_candidates = build_z_candidates(country_df)
        
        # Run linearity tests for all candidates
        print(f"Running linearity tests for {len(z_candidates)} candidates...")
        lm_results = []
        for z_name, z_series in z_candidates.items():
            result = lm_linearity_test_for_z(country_df, z_series, maxlags_hac=4)
            if result is not None:
                result['z_var'] = z_name
                lm_results.append(result)
        
        if len(lm_results) == 0:
            print(f"No valid linearity tests for {country_name}")
            continue
        
        lm_df = pd.DataFrame(lm_results).sort_values('p_value')
        str_linearity_tests[country_name] = lm_df
        
        # Select best z (lowest p-value)
        best_z_name = lm_df.iloc[0]['z_var']
        best_z = z_candidates[best_z_name]
        
        print(f"Best transition variable: {best_z_name} (p={lm_df.iloc[0]['p_value']:.4f})")
        
        # Build STR sample
        str_df = pd.DataFrame({
            "y": country_df["r_s"] - (country_df["i_for"] - country_df["i_dom"]),
            "rs_lag1": country_df["r_s"].shift(1),
            "eta_lag1": country_df["q"].shift(1),
            "z": best_z
        }).dropna()
        
        # Grid search with exponential gamma
        print(f"Running grid search...")
        gamma_grid = np.logspace(np.log10(0.125), np.log10(256.0), num=120)
        zc = pd.to_numeric(str_df["z"], errors="coerce").dropna().to_numpy()
        c_grid = np.quantile(zc, np.linspace(0.05, 0.95, 120))
        
        # Initialize and estimate
        str_model = STRModel(str_df, z_col='z')
        start_params = str_model.grid_search(gamma_grid, c_grid)
        
        print(f"Grid search complete. Starting NLS...")
        str_result = str_model.fit(start_params, hac_lags=4)
        
        # Store results
        str_results[country_name] = {
            'result': str_result,
            'z_name': best_z_name,
            'lm_pval': lm_df.iloc[0]['p_value']
        }
        
        print(f"\nSTR Results for {country_name}:")
        print(str_result.params)
        print(f"AIC: {str_result.aic:.2f}, BIC: {str_result.bic:.2f}")
        
    except Exception as e:
        print(f"ERROR processing {country_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\nSTR estimation complete for {len(str_results)} countries")

## 4. BUIP Estimation for All Countries

In [ ]:
from src.econometrics import BUIPModel
from src.utils import build_beh_sample

# Storage for BUIP results
buip_results = {}

for country_name, code in COUNTRIES.items():
    print(f"\n{'='*60}")
    print(f"BUIP: {country_name} ({code})")
    print(f"{'='*60}")
    
    try:
        # Get country data
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        
        if len(country_df) < 50:
            print(f"Insufficient data for {country_name}")
            continue
        
        # Build BUIP sample
        buip_df = build_beh_sample(country_df)
        
        print(f"Sample size: {len(buip_df)}")
        
        # Estimate with multistart
        print(f"Running multistart optimization...")
        buip_model = BUIPModel(buip_df)
        buip_result = buip_model.fit_multistart(n_starts=20, hac_lags=4)
        
        # Store results
        buip_results[country_name] = buip_result
        
        print(f"\nBUIP Results for {country_name}:")
        print(buip_result.params)
        print(f"AIC: {buip_result.aic:.2f}, BIC: {buip_result.bic:.2f}")
        
    except Exception as e:
        print(f"ERROR processing {country_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\nBUIP estimation complete for {len(buip_results)} countries")

## 5. Generate Publication Tables

In [ ]:
def format_with_stars(estimate, pval):
    """Add significance stars based on p-value"""
    if np.isnan(pval) or np.isnan(estimate):
        return ""
    
    stars = ""
    if pval < 0.01:
        stars = "***"
    elif pval < 0.05:
        stars = "**"
    elif pval < 0.10:
        stars = "*"
    
    if stars:
        return f"{estimate:.4f}{stars}"
    else:
        return f"{estimate:.4f}"

# STR Results Table
str_table_data = []
for country_name in COUNTRIES.keys():
    if country_name in str_results:
        res = str_results[country_name]['result']
        row = {'Country': country_name}
        for param in ['const', 'beta_c', 'beta_f', 'gamma', 'c', 'const1']:
            est = res.params.get(param, np.nan)
            pval = res.pvals.get(param, np.nan)
            row[param] = format_with_stars(est, pval)
        str_table_data.append(row)
    else:
        row = {'Country': country_name}
        for param in ['const', 'beta_c', 'beta_f', 'gamma', 'c', 'const1']:
            row[param] = "missing"
        str_table_data.append(row)

df_str_table = pd.DataFrame(str_table_data).set_index('Country')
print("\n=== STR MODEL RESULTS ===")
print(df_str_table)
df_str_table.to_csv(output_dir / "str_results_table.csv")

# BUIP Results Table
buip_table_data = []
for country_name in COUNTRIES.keys():
    if country_name in buip_results:
        res = buip_results[country_name]
        row = {'Country': country_name}
        for param in ['const', 'const1', 'beta_f', 'beta_c', 'gamma', 'c']:
            est = res.params.get(param, np.nan)
            pval = res.pvals.get(param, np.nan)
            row[param] = format_with_stars(est, pval)
        buip_table_data.append(row)
    else:
        row = {'Country': country_name}
        for param in ['const', 'const1', 'beta_f', 'beta_c', 'gamma', 'c']:
            row[param] = "missing"
        buip_table_data.append(row)

df_buip_table = pd.DataFrame(buip_table_data).set_index('Country')
print("\n=== BUIP MODEL RESULTS ===")
print(df_buip_table)
df_buip_table.to_csv(output_dir / "buip_results_table.csv")

print(f"\nTables saved to: {output_dir}")

## 6. Generate Aggregated Figures

In [ ]:
# Aggregate STR regime distribution
all_G_values = []
for country_name, result_dict in str_results.items():
    G = result_dict['result'].G.dropna()
    all_G_values.extend(G.values)

if len(all_G_values) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(all_G_values, bins=50, color='black', alpha=0.7, edgecolor='white')
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Threshold (G=0.5)')
    ax.set_xlabel('Transition Function G(z)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('STR Regime Distribution (All Countries)', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(output_dir / 'str_regime_distribution.png', dpi=300, bbox_inches='tight')
    plt.savefig(output_dir / 'str_regime_distribution.pdf', bbox_inches='tight')
    plt.show()
    print(f"STR distribution plot saved")

# Aggregate BUIP regime distribution
all_omega_values = []
for country_name, result in buip_results.items():
    omega = result.omega.dropna()
    all_omega_values.extend(omega.values)

if len(all_omega_values) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(all_omega_values, bins=50, color='darkblue', alpha=0.7, edgecolor='white')
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Threshold (ω=0.5)')
    ax.set_xlabel('Mixing Weight ω', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('BUIP Regime Distribution (All Countries)', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(output_dir / 'buip_regime_distribution.png', dpi=300, bbox_inches='tight')
    plt.savefig(output_dir / 'buip_regime_distribution.pdf', bbox_inches='tight')
    plt.show()
    print(f"BUIP distribution plot saved")

# Regime prevalence comparison
str_chartist_pct = (np.array(all_G_values) > 0.5).mean() * 100 if len(all_G_values) > 0 else 0
str_fundamentalist_pct = 100 - str_chartist_pct

buip_chartist_pct = (np.array(all_omega_values) < 0.5).mean() * 100 if len(all_omega_values) > 0 else 0
buip_fundamentalist_pct = 100 - buip_chartist_pct

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# STR
ax1.bar(['Chartist', 'Fundamentalist'], [str_chartist_pct, str_fundamentalist_pct], 
        color=['#e74c3c', '#3498db'], alpha=0.8, edgecolor='black')
ax1.set_ylabel('Prevalence (%)', fontsize=12)
ax1.set_title('STR Model Regime Prevalence', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 100)

# BUIP
ax2.bar(['Chartist', 'Fundamentalist'], [buip_chartist_pct, buip_fundamentalist_pct], 
        color=['#e74c3c', '#3498db'], alpha=0.8, edgecolor='black')
ax2.set_ylabel('Prevalence (%)', fontsize=12)
ax2.set_title('BUIP Model Regime Prevalence', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(output_dir / 'regime_prevalence_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig(output_dir / 'regime_prevalence_comparison.pdf', bbox_inches='tight')
plt.show()
print(f"Regime prevalence plot saved")

print(f"\n{'='*60}")
print("ALL FIGURES SAVED TO:", output_dir)
print(f"{'='*60}")

## 7. Summary Report

In [ ]:
print("\n" + "="*80)
print("REPRODUCTION PIPELINE COMPLETE")
print("="*80)

print(f"\nSTR Estimation:")
print(f"  - {len(str_results)} / {len(COUNTRIES)} countries estimated")
print(f"  - Results table: {output_dir / 'str_results_table.csv'}")

print(f"\nBUIP Estimation:")
print(f"  - {len(buip_results)} / {len(COUNTRIES)} countries estimated")
print(f"  - Results table: {output_dir / 'buip_results_table.csv'}")

print(f"\nOutput Files Generated:")
print(f"  - Descriptive statistics: descriptive_statistics.csv")
print(f"  - STR results: str_results_table.csv")
print(f"  - BUIP results: buip_results_table.csv")
print(f"  - STR regime distribution: str_regime_distribution.pdf/.png")
print(f"  - BUIP regime distribution: buip_regime_distribution.pdf/.png")
print(f"  - Regime prevalence comparison: regime_prevalence_comparison.pdf/.png")

print(f"\nAll files saved to: {output_dir}")
print("\n" + "="*80)